# Day 55 · Exercise 3: JWT Tokens

**What you'll build:** Implement `create_token(data, expires_in_minutes)` and `decode_token(token)` using `python-jose`. JWTs let the server verify identity without storing sessions — the signature proves the token hasn't been tampered with.

## Setup (provided)

In [ ]:
from jose import jwt, JWTError
from datetime import datetime, timedelta

SECRET_KEY = "test-secret-for-exercise"
ALGORITHM  = "HS256"


## Your Implementation

In [ ]:
def create_token(data: dict, expires_in_minutes: int = 60) -> str:
    """Encode a JWT with an expiry claim.

    Args:
        data:               Payload dict (e.g. {'user_id': 1, 'email': 'a@b.com'}).
        expires_in_minutes: Token lifetime in minutes.
    Returns:
        Signed JWT string (header.payload.signature).
    """
    # TODO: copy data, add 'exp' = datetime.utcnow() + timedelta(minutes=expires_in_minutes)
    #       return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)
    raise NotImplementedError

def decode_token(token: str) -> dict:
    """Decode and validate a JWT.

    Args:
        token: The JWT string to decode.
    Returns:
        Decoded payload dict.
    Raises:
        JWTError: If the signature is invalid or the token is expired.
    """
    # TODO: return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
    raise NotImplementedError


In [ ]:
def create_token(data: dict, expires_in_minutes: int = 60) -> str:
    payload = data.copy()
    payload["exp"] = datetime.utcnow() + timedelta(minutes=expires_in_minutes)
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

def decode_token(token: str) -> dict:
    return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])


## Check Your Work

In [ ]:
def _run_checks():
    score = 0
    total = 5

    def _chk(n, ok, msg):
        nonlocal score
        print(f"  {'✅' if ok else '❌'} Check {n}: {msg}")
        if ok:
            score += 1

    payload = {"user_id": 7, "email": "alice@example.com"}

    try:
        tok = create_token(payload)
    except NotImplementedError:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: create_token not implemented")
        print(f"\nScore: 0 / {total}")
        return

    _chk(1, isinstance(tok, str) and tok.count(".") == 2,
         f"create_token returns header.payload.signature (got {tok[:30]}...)")

    try:
        decoded = decode_token(tok)
    except NotImplementedError:
        for i in range(2, total + 1):
            print(f"  ❌ Check {i}: decode_token not implemented")
        print(f"\nScore: {score} / {total}")
        return

    _chk(2, isinstance(decoded, dict),
         f"decode_token returns a dict (got {type(decoded).__name__})")
    _chk(3, decoded.get("user_id") == 7 and decoded.get("email") == "alice@example.com",
         f"decoded payload has user_id=7 and email (got {decoded})")

    # expired token (negative lifetime = already expired)
    expired_tok = create_token({"user_id": 1}, expires_in_minutes=-1)
    try:
        decode_token(expired_tok)
        _chk(4, False, "expired token should raise JWTError")
    except JWTError:
        _chk(4, True, "expired token raises JWTError ✓")
    except Exception as e:
        _chk(4, False, f"expired token raised {type(e).__name__} instead of JWTError")

    # tampered token
    parts = tok.split(".")
    tampered = parts[0] + "." + parts[1] + ".INVALIDSIG"
    try:
        decode_token(tampered)
        _chk(5, False, "tampered token should raise JWTError")
    except JWTError:
        _chk(5, True, "tampered token raises JWTError ✓")
    except Exception as e:
        _chk(5, False, f"tampered token raised {type(e).__name__} instead of JWTError")

    print(f"\nScore: {score} / {total}")
    if score == total:
        print("🎉 Exercise complete!")

_run_checks()


## Bonus Challenge

Decode a token manually: split on `.`, base64-decode the middle part (pad with `=` to a multiple of 4 bytes), and `json.loads` it. You'll see your payload plus the `exp` timestamp. This shows why the payload is readable without the secret key — JWT is *signed*, not encrypted.

## Solution

<details>
<summary>Show solution</summary>

```python
def create_token(data: dict, expires_in_minutes: int = 60) -> str:
    payload = data.copy()
    payload["exp"] = datetime.utcnow() + timedelta(minutes=expires_in_minutes)
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

def decode_token(token: str) -> dict:
    return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
```

**Why this works:** `jwt.encode` creates the three-part JWT: base64(header) +
base64(payload) + HMAC-SHA256(header+payload, SECRET_KEY). `jwt.decode` verifies
the signature and checks the `exp` claim automatically — an expired or tampered
token raises `JWTError` before you see the payload. The secret key must stay
secret; anyone with it can mint valid tokens.

</details>